# Customer Segmentation: RFM + K-Means Analysis

**Author**: Cayman Roden  
**Prerequisite**: Run `01_eda_cleaning.ipynb` first (builds `data/processed/olist_master.parquet`)

---

## Overview

Segments Olist customers into 4 behavioral groups using:
1. **RFM scoring** (Recency, Frequency, Monetary) on `customer_unique_id`
2. **K-Means clustering** (k=4, validated by elbow + silhouette + academic precedent)
3. **Business-meaningful labeling** (Champions, Loyal, At-Risk, Hibernating)

**Methodology**: Log1p transform → StandardScaler → KMeans++ initialization.  
Academic basis: Rousseeuw (1987) silhouette; k=4 validated in peer-reviewed Olist study.

---

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sys.path.insert(0, str(Path.cwd()))
from src.data_loader import load_master
from src.ml_pipeline import (
    compute_rfm, preprocess_rfm, elbow_scores,
    silhouette_scores, run_clustering, label_clusters
)

warnings.filterwarnings('ignore')
pio.templates.default = 'plotly_white'
BLUE, GREEN, AMBER, RED = '#2563EB', '#10B981', '#F59E0B', '#EF4444'
SEGMENT_COLORS = {
    'Champions': GREEN, 'Loyal Customers': BLUE,
    'At-Risk': AMBER, 'Hibernating': RED
}
print("Environment ready.")

---
## 1 · Load Data & Compute RFM

In [ ]:
df = load_master()
print(f"Loaded: {len(df):,} rows, {df['customer_unique_id'].nunique():,} unique customers")

rfm = compute_rfm(df)
print(f"RFM table: {len(rfm):,} customers")
print(rfm.describe().round(2))

In [ ]:
# RFM raw distributions — show the right-skew problem
fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Recency (days)', 'Frequency (orders)', 'Monetary (R$)'])

for i, col in enumerate(['recency_days', 'frequency', 'monetary'], 1):
    fig.add_trace(
        go.Histogram(x=rfm[col], nbinsx=50, marker_color=BLUE, opacity=0.8, showlegend=False),
        row=1, col=i
    )
fig.update_layout(
    title='RFM Raw Distributions (heavy right skew — log transform needed)',
    height=320, margin=dict(l=0, r=0, t=60, b=0)
)
fig.show()

for col in ['recency_days', 'frequency', 'monetary']:
    print(f"{col}: skewness = {rfm[col].skew():.2f}")

**Observation**: All three RFM dimensions show strong right skew. K-Means is distance-based and sensitive to scale — log1p transform is required before standardizing.

---
## 2 · Log1p Transform & Standardize

In [ ]:
scaled, rfm_log = preprocess_rfm(rfm)

fig = make_subplots(rows=2, cols=3,
    subplot_titles=[
        'Recency (raw)', 'Frequency (raw)', 'Monetary (raw)',
        'Recency (log1p)', 'Frequency (log1p)', 'Monetary (log1p)'
    ])

for i, col in enumerate(['recency_days', 'frequency', 'monetary'], 1):
    fig.add_trace(go.Histogram(x=rfm[col], nbinsx=40, marker_color=RED,
                               opacity=0.7, showlegend=False), row=1, col=i)
    fig.add_trace(go.Histogram(x=rfm_log[col], nbinsx=40, marker_color=GREEN,
                               opacity=0.7, showlegend=False), row=2, col=i)

fig.update_layout(
    title='RFM: Before (red) vs After Log1p Transform (green)',
    height=480, margin=dict(l=0, r=0, t=60, b=0)
)
fig.show()

print("Post-transform skewness:")
for col in ['recency_days', 'frequency', 'monetary']:
    print(f"  {col}: {rfm_log[col].skew():.3f}")

---
## 3 · Select Optimal K

Three validation methods — look for agreement:
- **Elbow**: inertia (WCSS) vs k — bend = optimal
- **Silhouette**: cohesion vs separation — higher = better
- **Expected result**: k=4 (peer-reviewed Olist validation)

In [ ]:
k_range = range(2, 9)
elbow = elbow_scores(scaled, k_range)
sil = silhouette_scores(scaled, k_range)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Elbow Method (Inertia)', 'Silhouette Score (higher = better)'])

fig.add_trace(go.Scatter(
    x=list(elbow.keys()), y=list(elbow.values()),
    mode='lines+markers', marker=dict(size=8, color=BLUE),
    line=dict(color=BLUE, width=2), showlegend=False), row=1, col=1)

fig.add_trace(go.Scatter(
    x=list(sil.keys()), y=list(sil.values()),
    mode='lines+markers', marker=dict(size=8, color=GREEN),
    line=dict(color=GREEN, width=2), showlegend=False), row=1, col=2)

fig.add_vline(x=4, line_dash='dash', line_color=RED, row=1, col=1)
fig.add_vline(x=4, line_dash='dash', line_color=RED, row=1, col=2)

fig.update_xaxes(title_text='Number of Clusters (k)')
fig.update_yaxes(title_text='Inertia', row=1, col=1)
fig.update_yaxes(title_text='Silhouette Score', row=1, col=2)
fig.update_layout(
    height=360, margin=dict(l=0, r=0, t=60, b=0),
    title='Cluster Selection — red line = k=4 (selected)'
)
fig.show()

best_sil = max(sil, key=sil.get)
print("Silhouette scores:")
for k, s in sil.items():
    tag = " <- selected (k=4)" if k == 4 else (f" <- best silhouette" if k == best_sil and best_sil != 4 else "")
    print(f"  k={k}: {s:.4f}{tag}")
print(f"Note: Scores 0.3-0.5 are normal for real customer data.")

---
## 4 · Fit K-Means (k=4)

In [ ]:
km, labels = run_clustering(scaled, k=4)
rfm_labeled = label_clusters(rfm, labels)

dist = rfm_labeled['segment'].value_counts()
print("Segment sizes:")
for seg, n in dist.items():
    print(f"  {seg}: {n:,} customers ({n/len(rfm_labeled):.1%})")

In [ ]:
profiles = (
    rfm_labeled.groupby('segment')[['recency_days', 'frequency', 'monetary']]
    .mean().round(2).reset_index()
)
print("Mean RFM per segment:")
print(profiles.to_string(index=False))

---
## 5 · Visualize Clusters

In [ ]:
# PCA 2D projection for visualization
pca = PCA(n_components=2, random_state=42)
rfm_log_arr = np.log1p(rfm_labeled[['recency_days', 'frequency', 'monetary']])
scaled_full = StandardScaler().fit_transform(rfm_log_arr)
coords = pca.fit_transform(scaled_full)

rfm_plot = rfm_labeled.copy()
rfm_plot['PC1'] = coords[:, 0]
rfm_plot['PC2'] = coords[:, 1]

sample = rfm_plot.sample(min(3000, len(rfm_plot)), random_state=42)

var_explained = sum(pca.explained_variance_ratio_[:2])
fig = px.scatter(
    sample, x='PC1', y='PC2', color='segment',
    color_discrete_map=SEGMENT_COLORS,
    opacity=0.5,
    labels={
        'PC1': f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)",
        'PC2': f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)"
    },
    title=f"Customer Segments — PCA Projection (captures {var_explained:.1%} of variance)"
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(legend=dict(orientation='h', y=1.1), height=440,
                  margin=dict(l=0, r=0, t=60, b=0))
fig.show()

In [ ]:
# Segment profiles: normalized grouped bar
normalized = profiles.copy()
for col in ['recency_days', 'frequency', 'monetary']:
    mn, mx = normalized[col].min(), normalized[col].max()
    normalized[col] = (normalized[col] - mn) / (mx - mn + 1e-9)
normalized['recency_days'] = 1 - normalized['recency_days']  # invert: recent = better

melted = normalized.melt(
    id_vars='segment',
    value_vars=['recency_days', 'frequency', 'monetary'],
    var_name='metric', value_name='score'
)
melted['metric'] = melted['metric'].map({
    'recency_days': 'Recency (inverted)',
    'frequency': 'Frequency',
    'monetary': 'Monetary'
})

fig = px.bar(
    melted, x='metric', y='score', color='segment', barmode='group',
    color_discrete_map=SEGMENT_COLORS,
    labels={'score': 'Normalized Score (0=worst, 1=best)', 'metric': ''},
    title='Segment Profiles — Normalized RFM Scores'
)
fig.update_layout(yaxis_range=[0, 1.15], height=380,
                  legend=dict(orientation='h', y=1.1),
                  margin=dict(l=0, r=0, t=60, b=0))
fig.show()

In [ ]:
# Segment distribution pie
dist_df = rfm_labeled['segment'].value_counts().reset_index()
dist_df.columns = ['segment', 'count']

fig = px.pie(dist_df, names='segment', values='count',
             color='segment', color_discrete_map=SEGMENT_COLORS,
             title='Customer Segment Distribution')
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(showlegend=False, height=380,
                  margin=dict(l=0, r=0, t=60, b=0))
fig.show()

---
## 6 · Business Implications

**Champions** (high R, high F, high M)
- VIP program; early product access; referral asks
- Highest LTV — most important to retain

**Loyal Customers** (medium R, high F, medium M)
- Cross-sell into adjacent categories
- Loyalty rewards to prevent downgrade to At-Risk

**At-Risk** (haven't purchased recently, medium F+M)
- Time-sensitive win-back within 30-day window
- Personalized discount based on previously purchased categories

**Hibernating** (high R, low F, low M)
- One-time strong win-back offer
- If no response in 90 days, remove from active marketing budget

In [ ]:
# Export segment assignments for dashboard
output_path = Path('data/processed/rfm_segments.parquet')
rfm_labeled.to_parquet(output_path, index=False)
print(f"Saved: {output_path}  ({len(rfm_labeled):,} rows)")
print(rfm_labeled['segment'].value_counts().to_string())